In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 1. LOAD MODEL ĐÃ TRAIN
# =============================================================================
model_path = 'DA\pkl_file\task_1_PB_RF.pkl'
print(f"⏳ Đang load model từ {model_path}...")

try:
    model_data = joblib.load(model_path)
    pipeline = model_data['pipeline']
    
    # Lấy danh sách tên các cột từ lúc train
    audio_cols = model_data['audio_features']
    linguistic_cols = model_data.get('linguistic_cols', ['sentiment_score', 'noun_ratio', 'verb_ratio', 'adj_ratio'])
    
    print("   ✓ Load thành công!")
except Exception as e:
    print(f"❌ Lỗi load file: {e}")
    exit()

# =============================================================================
# 2. TÁI TẠO DANH SÁCH TÊN FEATURES (INPUT BAN ĐẦU)
# =============================================================================
# Thứ tự lúc ghép (hstack) trong file train là: [Audio] + [Linguistic] + [PhoBERT]
feature_names = []

# 1. Audio Features
feature_names.extend(audio_cols)

# 2. Linguistic Features
feature_names.extend(linguistic_cols)

# 3. PhoBERT Features (768 chiều)
# Vì PhoBERT là vector ẩn, ta đặt tên là phobert_0, phobert_1...
feature_names.extend([f'phobert_dim_{i}' for i in range(768)])

feature_names = np.array(feature_names)
print(f"   ∑ Tổng số features ban đầu: {len(feature_names)}")

# =============================================================================
# 3. XỬ LÝ FEATURE SELECTION (LẤY NHỮNG CỘT ĐƯỢC GIỮ LẠI)
# =============================================================================
# Lấy bước 'feature_selection' trong pipeline
try:
    selector = pipeline.named_steps['feature_selection']
    # get_support() trả về mảng True/False (True là được giữ lại)
    selected_mask = selector.get_support()
    
    # Lọc danh sách tên feature theo mask này
    selected_feature_names = feature_names[selected_mask]
    
    print(f"   ✂️  Số features sau khi lọc (SelectFromModel): {len(selected_feature_names)}")
except KeyError:
    print("   ⚠️ Không tìm thấy bước 'feature_selection', lấy toàn bộ features.")
    selected_feature_names = feature_names

# =============================================================================
# 4. LẤY ĐỘ QUAN TRỌNG (IMPORTANCE) TỪ RANDOM FOREST
# =============================================================================
classifier = pipeline.named_steps['classifier']

if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
else:
    print("❌ Model không hỗ trợ feature_importances_")
    exit()

# Tạo DataFrame
df_imp = pd.DataFrame({
    'Feature': selected_feature_names,
    'Importance': importances
})

# Sắp xếp giảm dần
df_imp = df_imp.sort_values(by='Importance', ascending=False).reset_index(drop=True)

# =============================================================================
# 5. XUẤT RA CSV VÀ VẼ BIỂU ĐỒ
# =============================================================================
# Lưu ra file CSV để bạn tiện xem
output_csv = 'feature_importance_ranking.csv'
df_imp.to_csv(output_csv, index=False)
print(f"   ✓ Đã lưu bảng xếp hạng vào: {output_csv}")

# Hiển thị Top 20 features quan trọng nhất
print("\n🏆 TOP 20 FEATURES QUAN TRỌNG NHẤT:")
print(df_imp.head(20))

# Vẽ biểu đồ
plt.figure(figsize=(12, 8))
sns.barplot(data=df_imp.head(20), x='Importance', y='Feature', palette='viridis')
plt.title('Top 20 Feature Importances (Random Forest)')
plt.xlabel('Mức độ quan trọng')
plt.ylabel('Tên đặc trưng')
plt.tight_layout()
plt.show()

# =============================================================================
# 6. PHÂN TÍCH NHANH
# =============================================================================
# Đếm xem trong Top 50 có bao nhiêu cái thuộc về Audio, Linguistic, hay PhoBERT
top_50 = df_imp.head(50)['Feature'].tolist()

n_audio = sum(1 for f in top_50 if f in audio_cols)
n_ling = sum(1 for f in top_50 if f in linguistic_cols)
n_bert = sum(1 for f in top_50 if 'phobert_dim_' in f)

print("\n📊 THỐNG KÊ TRONG TOP 50 QUAN TRỌNG NHẤT:")
print(f"   - Audio Features:      {n_audio}")
print(f"   - Linguistic (Ngữ pháp): {n_ling}")
print(f"   - PhoBERT (Ý nghĩa):   {n_bert}")

<>:10: SyntaxWarning: invalid escape sequence '\p'
<>:10: SyntaxWarning: invalid escape sequence '\p'
C:\Users\huyen\AppData\Local\Temp\ipykernel_13536\4227796850.py:10: SyntaxWarning: invalid escape sequence '\p'
  model_path = 'DA\pkl_file\task_1_PB_RF.pkl'


⏳ Đang load model từ DA\pkl_file	ask_1_PB_RF.pkl...
❌ Lỗi load file: [Errno 22] Invalid argument: 'DA\\pkl_file\task_1_PB_RF.pkl'


C:\Users\huyen\AppData\Local\Temp\ipykernel_13536\4227796850.py:10: SyntaxWarning: invalid escape sequence '\p'
  model_path = 'DA\pkl_file\task_1_PB_RF.pkl'


NameError: name 'audio_cols' is not defined

: 

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Sklearn Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

# Models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering

# Metrics
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, f1_score, silhouette_score

warnings.filterwarnings('ignore')

# Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (16, 10), 'font.size': 12})

# =============================================================================
# 1. LOAD & PREPARE DATA
# =============================================================================
print("⏳ Đang tải dữ liệu...")
try:
    df = pd.read_csv('mergerd_balanced_and_features.csv')
except FileNotFoundError:
    # Fallback cho đường dẫn Windows nếu cần
    df = pd.read_csv(r'final_data\merged_balanced_and_features.csv')

# Loại bỏ các cột không dùng để train
cols_to_drop = [
    'file_name', 'title', 'artists', 'spotify_release_date', 'spotify_genres',
    'is_hit', 'total_plays', 'spotify_streams', 'spotify_popularity',
    'sentiment', 'sentiment_score', 'sentiment_confidence',
    'Unnamed: 0', 'hit', 'lyric', 'clean_lyric', 'primary_genre', 'cluster', 'musical_key'
]
# Tự động lấy các cột số (Audio features)
numeric_feats = [c for c in df.columns if c not in cols_to_drop and pd.api.types.is_numeric_dtype(df[c])]

# Xử lý Genre (cho P4)
def extract_primary_genre(genre_str):
    try:
        if isinstance(genre_str, str):
            genres = [g.strip() for g in genre_str.replace('[', '').replace(']', '').replace("'", "").replace('"', "").split(',')]
            if len(genres) > 0 and genres[0] != '': return genres[0]
        return "unknown"
    except: return "unknown"

df['primary_genre'] = df['spotify_genres'].apply(extract_primary_genre)

# Pipeline tiền xử lý chung
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Dictionaries để lưu kết quả
results_p1 = {}
results_p3 = {}
results_p4 = {}
results_p2 = {}

# =============================================================================
# 2. CHẠY THỬ NGHIỆM (TOURNAMENT)
# =============================================================================

print("⚡ Đang chạy thử nghiệm các mô hình...")

# --- P1: Spotify Popularity (Regression) ---
print("   🔹 P1: Đánh giá mô hình Hồi quy (Popularity)...")
df_p1 = df.dropna(subset=['spotify_popularity'])
if len(df_p1) > 20:
    X = df_p1[numeric_feats]
    y = df_p1['spotify_popularity']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models_reg = {
        'Ridge': Ridge(),
        'RandomForest': RandomForestRegressor(n_estimators=50, random_state=42),
        'GradientBoosting': GradientBoostingRegressor(random_state=42),
        'SVR': SVR()
    }

    for name, model in models_reg.items():
        pipe = Pipeline(steps=[('pre', preprocessor), ('model', model)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        results_p1[name] = r2_score(y_test, y_pred)

# --- P3: Sentiment (Classification) ---
print("   🔹 P3: Đánh giá mô hình Phân loại (Sentiment)...")
df_p3 = df.dropna(subset=['sentiment'])
df_p3 = df_p3[df_p3['sentiment'].isin(['positive', 'negative', 'neutral'])]
if len(df_p3) > 20:
    X = df_p3[numeric_feats]
    y = df_p3['sentiment']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    models_clf = {
        'LogisticReg': LogisticRegression(max_iter=500),
        'RandomForest': RandomForestClassifier(n_estimators=50, random_state=42),
        'GradientBoosting': GradientBoostingClassifier(random_state=42),
        'SVC': SVC(),
        'KNN': KNeighborsClassifier()
    }

    for name, model in models_clf.items():
        pipe = Pipeline(steps=[('pre', preprocessor), ('model', model)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        results_p3[name] = accuracy_score(y_test, y_pred)

# --- P4: Genre (Classification) ---
print("   🔹 P4: Đánh giá mô hình Phân loại (Genre)...")
top_genres = df['primary_genre'].value_counts().head(5).index.tolist()
if 'unknown' in top_genres: top_genres.remove('unknown')
df_p4 = df[df['primary_genre'].isin(top_genres)]

if len(df_p4) > 20:
    X = df_p4[numeric_feats]
    y = df_p4['primary_genre']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    for name, model in models_clf.items():
        pipe = Pipeline(steps=[('pre', preprocessor), ('model', model)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        results_p4[name] = accuracy_score(y_test, y_pred)

# --- P2: Clustering (Unsupervised) ---
print("   🔹 P2: Đánh giá Phân cụm (Clustering)...")
X_p2 = df[numeric_feats]
X_processed = preprocessor.fit_transform(X_p2)
# Dùng PCA giảm chiều để tính toán nhanh hơn cho việc so sánh
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

for k in range(2, 6):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    results_p2[f'KMeans (k={k})'] = silhouette_score(X_pca, labels)

# =============================================================================
# 3. VẼ BIỂU ĐỒ & XUẤT BÁO CÁO
# =============================================================================

print("\n📊 Đang vẽ biểu đồ...")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
plt.subplots_adjust(hspace=0.3, wspace=0.2)

# Hàm vẽ bar chart
def plot_bar(results, ax, title, ylabel, color):
    if results:
        names = list(results.keys())
        scores = list(results.values())
        sns.barplot(x=names, y=scores, ax=ax, palette=color)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, max(scores) * 1.2 if max(scores) > 0 else 1.0)
        # Thêm số lên đầu cột
        for i, v in enumerate(scores):
            ax.text(i, v + 0.01, f"{v:.3f}", ha='center', va='bottom', fontweight='bold')
    else:
        ax.text(0.5, 0.5, "Không đủ dữ liệu", ha='center', va='center')

# Vẽ 4 biểu đồ
plot_bar(results_p1, axes[0, 0], 'P1: Dự đoán Độ phổ biến (R2 Score)', 'R2 Score', 'Blues_d')
plot_bar(results_p3, axes[0, 1], 'P3: Phân tích Cảm xúc (Accuracy)', 'Accuracy', 'Greens_d')
plot_bar(results_p4, axes[1, 0], 'P4: Phân loại Thể loại (Accuracy)', 'Accuracy', 'Reds_d')
plot_bar(results_p2, axes[1, 1], 'P2: Phân cụm Phong cách (Silhouette Score)', 'Silhouette Score', 'Purples_d')

# Lưu ảnh
output_img = 'model_comparison_report.png'
plt.savefig(output_img, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Đã lưu biểu đồ so sánh vào file: {output_img}")

# --- IN BÁO CÁO TEXT ---
print("\n" + "="*50)
print("📝 BÁO CÁO TỔNG HỢP KẾT QUẢ HUẤN LUYỆN")
print("="*50)

def print_best(name, results, metric_name):
    if not results:
        print(f"\n{name}: Không có dữ liệu.")
        return
    best_model = max(results, key=results.get)
    best_val = results[best_model]
    print(f"\n📌 {name}")
    print(f"   🏆 Model tốt nhất: {best_model}")
    print(f"   💎 {metric_name}: {best_val:.4f}")
    
    # Logic nhận xét tự động
    if metric_name == 'R2 Score':
        cmt = "Khá tốt" if best_val > 0.5 else "Thấp (Cần thêm Lyrics/Artist)"
    elif metric_name == 'Accuracy':
        cmt = "Xuất sắc" if best_val > 0.8 else ("Tốt" if best_val > 0.6 else "Trung bình")
    else:
        cmt = "Phân cụm rõ ràng" if best_val > 0.5 else "Các cụm còn chồng lấn"
    print(f"   💡 Nhận xét: {cmt}")

print_best("P1 - Dự đoán Popularity", results_p1, "R2 Score")
print_best("P3 - Phân tích Cảm xúc", results_p3, "Accuracy")
print_best("P4 - Phân loại Thể loại", results_p4, "Accuracy")
print_best("P2 - Phân cụm Phong cách", results_p2, "Silhouette Score")
print("\n" + "="*50)

⏳ Đang tải dữ liệu...


FileNotFoundError: [Errno 2] No such file or directory: 'final_data\\merged_balanced_and_features.csv'